# Contrastive Ecoacoustic Indices 
## A Comprehensive Notebook

This notebook provides an example of code using our computational pipeline defining novel Contrastive Ecoacoustic Indices (CEI).

Exploiting the strength and flexibility offered by contrastive inference through the Contrastive Language-Audio Pretraining ([CLAP](https://github.com/microsoft/CLAP)) model, we define CEI as new acoustic indices that allow to globally characterize monitored soundscapes into four categorical constituents:
* __Biophony__, *i.e.*, sounds produced by living organisms (but humans) in the environment;
* __Geophony__, *i.e.*, sounds produced by all abiotic elements in the environment;
* __Anthropophony__, *i.e.*, sounds produced directly by humans via their phonation or body;
* __Technophony__, *i.e.*, sounds produced by all machines and technologies from human activity.

## First things first

By default, the audio examples are taken from a small amount of data collected in the Risoux forest, Jura, France (git folder `data`) and the descriptions to investigate the soundscape content are the ones defined in our file `txt/classes.txt`. 

A configuration file, containing various hyperparameter is also required and goes by default to our setup located at `src/config.txt`. 

The user of this notebook can easily change these audio data and text descriptions or update configuration:

In [ ]:
audio_path = '../data/'
texts = '../txt/classes.txt'
config_file = "../src/config.txt"

In [ ]:
import os
assert os.path.exists(audio_path)
assert os.path.exists(texts)
assert os.path.exists(config_file)

In [ ]:
import librosa 
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 
import soundfile as sf
import torch 

from msclap import CLAP 
from re import split,match
from unidecode import unidecode

In [ ]:
def progress_bar(current_step, total_steps):
    assert total_steps>=current_step
    bar_length = 50
    progress = current_step / total_steps
    block = int(round(bar_length * progress))
    bar = '|' + '█' * block + ' ' * (bar_length - block) + '|'
    percentage = round(progress * 100, 2)
    print(f'\r{bar} {percentage}% ({current_step}/{total_steps}) Complete ', 
          end='', flush=True)

### CLAP configuration

In [ ]:
# Configuration hyperparameters
conv = { 'int': int, 
         'float': float, 
         'bool': lambda x: x=='True', 
         'str': str }
args = {}
with open(config_file,'r') as config:
    lines=config.readlines() 
    for line in lines:
        if len(line) and line!='\n' and line[0]!='#':
            while "\n" in line:
                line = line.replace("\n",'')
            p_name, p_value, p_type = split(',', line)
            args[p_name] = conv[p_type](p_value)

In [ ]:
eps = args['eps']
facecolor = args['facecolor']
grey_color = args['grey_color']
axbackcolor = args['axbackcolor']

In [ ]:
# CLAP model 
clap_model = CLAP(version=args["clap_version"], use_cuda=args["clap_cuda"])
assert clap_model

### Audios

In [ ]:
# Select the audio data
audio_list = [ os.path.join(audio_path, audio) for audio in os.listdir(audio_path)]
audio_list.sort()
assert len(audio_list)

In [ ]:
 # Temporary folder for short audio segments
tmp_audio = args["tmp_audio"]
if os.path.exists(tmp_audio):
    os.system(f"rm -r {tmp_audio}")
os.system(f"mkdir {tmp_audio}")

In [ ]:
# Extract audio embeddings
sub_audios = []
emb_dur = args["duration"]
N_BATCHES=len(audio_list)//args["batch_size"] 
audio_embeddings = None

# Iterate over all audios per batch
for i in range(1+N_BATCHES):
    # Retrieve unprocessed audios
    progress_bar(1+i, 1+N_BATCHES)
    
    if i==N_BATCHES:
        audios_i = audio_list[i*args["batch_size"]:]
    else:
        audios_i = audio_list[i*args["batch_size"]:(i+1)*args["batch_size"]]

    # Excerpts of short duration
    for audio in audios_i:
        audio_name = split('/', audio)[-1][:-4]
        try:
            y, sr = librosa.load(audio, sr=None)
        except Exception as e:
            print(f"An exception {e} occurred for the audio: {audio}. Ignored.")
            continue 
        n_emb = round((len(y)/sr)/emb_dur)
        for j in range(n_emb):
            # Temporary short audio files
            end_j = min(len(y), int(((j+1)*emb_dur)*sr))
            y_j = y[int((j*emb_dur)*sr):end_j]
            sf.write(f"{tmp_audio}{audio_name}_{j+1}_{n_emb}.wav", y_j, sr)

            # Keep track of the sub-divisions per file
            sub_audios.append(audio)

    the_audios_i = [ tmp_audio+f for f in os.listdir(tmp_audio) ]
    the_audios_i.sort()
    if not len(the_audios_i):
        print("Empty audio batch. Ignored.")
        continue

    # Extract audio embeddings
    audio_emb_i = clap_model.get_audio_embeddings(the_audios_i)
    if audio_embeddings is None:
        audio_embeddings = audio_emb_i 
    else:
        audio_embeddings = torch.cat((audio_embeddings, audio_emb_i), dim=0)

    # Delete all small audio excerpts
    os.system(f"rm {tmp_audio}*")

In [ ]:
# Get rid of temporary folder
os.system(f"rm -r {tmp_audio}")

### Texts and similarities

In [ ]:
# Select text data
all_classes = open(texts, "r").read()
raw_classes = [ unidecode(c.lower().replace(' - ', ' '))
                for c in split('\n', all_classes) if len(c) ]
classes = [ unidecode(split(' - ', c.lower())[-1]) # only the finest sub-class
            for c in split('\n', all_classes) if len(c) ]

In [ ]:
# Extract text embeddings
text_embeddings = clap_model.get_text_embeddings(classes)

In [ ]:
# Compute similarity between audio and text embeddings 
similarities = clap_model.compute_similarity(audio_embeddings, 
                                             text_embeddings)
similarities = similarities.detach().numpy()

### Defining Contrastive Ecoacoustic Indices (CEI)

In [ ]:
# Global categories
biophony = np.array([ ("biophony" in c) for c in raw_classes ])
geophony = np.array([ ("geophony" in c) for c in raw_classes ])
antphony = np.array([ ("anthropophony" in c) for c in raw_classes ])
tecphony = np.array([ ("technophony" in c) for c in raw_classes ])

In [ ]:
# Activation matrix
activation_matrix = (similarities>args["thr_sim"])*similarities
activation_matrix = np.maximum([eps], np.minimum([1.], activation_matrix/args["max_sim"]))

assert activation_matrix.shape[0]==len(sub_audios)
df_am = pd.DataFrame(activation_matrix)
df_am.columns = classes
df_am.index = sub_audios

In [ ]:
#  Per-time activation of the different classes
alphas = activation_matrix > args["thr_activity"]

alphas_b = alphas[:,biophony].sum(axis=-1)
alphas_b = alphas_b/(eps+alphas_b.max())

alphas_g = alphas[:,geophony].sum(axis=-1)
alphas_g = alphas_g/(eps+alphas_g.max())

alphas_a = alphas[:,antphony].sum(axis=-1)
alphas_a = alphas_a/(eps+alphas_a.max())

alphas_t = alphas[:,tecphony].sum(axis=-1)
alphas_t = alphas_t/(eps+alphas_t.max())

In [ ]:
# High-level temporal curves per categories
bio_m = activation_matrix[:,biophony].mean(axis=-1)
geo_m = activation_matrix[:,geophony].mean(axis=-1)
ant_m = activation_matrix[:,antphony].mean(axis=-1)
tec_m = activation_matrix[:,tecphony].mean(axis=-1)
bio = np.ma.masked_array(activation_matrix[:,biophony], mask=~alphas[:,biophony]).mean(axis=-1).filled(0.)
geo = np.ma.masked_array(activation_matrix[:,geophony], mask=~alphas[:,geophony]).mean(axis=-1).filled(0.)
ant = np.ma.masked_array(activation_matrix[:,antphony], mask=~alphas[:,antphony]).mean(axis=-1).filled(0.)
tec = np.ma.masked_array(activation_matrix[:,tecphony], mask=~alphas[:,tecphony]).mean(axis=-1).filled(0.)

In [ ]:
# Temporal curves per categories (for display only)
bio = np.maximum(bio_m, np.minimum([1.], (1+alphas_b)*bio))
geo = np.maximum(geo_m, np.minimum([1.], (1+alphas_g)*geo))
ant = np.maximum(ant_m, np.minimum([1.], (1+alphas_a)*ant))
tec = np.maximum(tec_m, np.minimum([1.], (1+alphas_t)*tec))

In [ ]:
# Data frames
df = pd.DataFrame([sub_audios, bio, geo, ant, tec]).T
df.columns = ['filenames', 'biophony', 'geophony', 'anthropophony', 'technophony']
df = df.groupby('filenames').mean() 

In [ ]:
# Contrastive Ecoacoustic Indices (CEI)
cei = df.median()
print(cei)

### Displays

In [ ]:
# Activation matrix 
plt.close('all')
interp = 'nearest' 
plt.figure(figsize=(10,6), facecolor=facecolor)
plt.gca().xaxis.label.set_color(grey_color)
plt.gca().yaxis.label.set_color(grey_color)
plt.gca().tick_params(axis='x', colors=grey_color)
plt.gca().tick_params(axis='y', colors=grey_color)
plt.gca().title.set_color(grey_color)
plt.gca().spines['top'].set_color(grey_color)
plt.gca().spines['bottom'].set_color(grey_color)
plt.gca().spines['left'].set_color(grey_color)
plt.gca().spines['right'].set_color(grey_color)
plt.imshow(df_am.T.to_numpy()[1:].astype(float), cmap='magma', aspect='auto', interpolation=interp)
cbar = plt.colorbar()
cbar.ax.yaxis.set_ticklabels([str(v/10.) for v in range(0,11,2) ], color=grey_color)
cbar.ax.yaxis.set_tick_params(color=grey_color)
plt.clim([0,1])
plt.yticks(np.arange(len(classes)), classes, ha='right', fontsize=4)
plt.xticks([]) 
plt.xlabel('Time', fontsize=10)
plt.ylabel('Classes', fontsize=10)
plt.title(f'Soundscape - raw contrastive activations', fontsize=10)

In [ ]:
# Contrastive curves
plt.close('all')
fig, axes = plt.subplots(4, 1, figsize=(8, 10), facecolor=facecolor)
raw_colors = ['green', 'blue', 'orange', 'red']
colors = ["#66CC66", "#88B4E0", "#FFAA66", "#FF6666"]
values = [ np.array(df[col]) for col in df.columns ]
ceis = [ np.median(v) for v in values ]
titles = ['Biophony', 'Geophony', 'Anthropophony', 'Technophony']
for i in range(len(axes)):
    x = np.arange(values[i].shape[0])
    axes[i].plot(x, values[i], color=colors[i])
    axes[i].spines['top'].set_color(grey_color)
    axes[i].spines['bottom'].set_color(grey_color)
    axes[i].spines['left'].set_color(grey_color)
    axes[i].spines['right'].set_color(grey_color)
    axes[i].set_facecolor(axbackcolor)
    axes[i].set_title(titles[i], color=colors[i], fontsize=10)
    axes[i].tick_params(axis='x', colors=grey_color)
    axes[i].tick_params(axis='y', colors=grey_color)
    axes[i].xaxis.set_visible(False)
    axes[i].set_ylim(-0.05, 1.05)
    axes[i].axhline(y=0.5, color=colors[i], linestyle=':', alpha=0.8)
    axes[i].axhline(y=ceis[i], color=raw_colors[i], linestyle='--', alpha=0.7)
fig.text(0.5, 0.025, "Time", ha='center', color=grey_color, fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle(f'Soundscape contrastive curve \n [%] bio={np.round(ceis[0], 2)} geo={np.round(ceis[1], 2)} ant={np.round(ceis[2], 2)} tec={np.round(ceis[3], 2)}', color=grey_color, fontsize=10)

### Unit test

In [ ]:
# Reproducibility: max 0.5% relative error 
tolerance = 0.005
ref = pd.read_csv("./cei_unit_test.csv")
relative_error = cei.values-ref.to_numpy()[:,-1]
assert np.all(relative_error <= tolerance)